In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.fft as fft
import seaborn as sns
import torch
from scipy.stats import genextreme, genpareto, pareto
from wxderivs import (
    DIR_WORK,
    MONTH_CDD,
    MONTH_HDD,
    SET_CDD,
    SET_HDD,
    SET_USA,
    dct_name_file,
    fit_mean_std,
)

sys.path.append(str(DIR_WORK / "unibm"))

import unibm
import unibm.benchmark

print(dct_name_file)


{WindowsPath('c:/Users/saiki/OneDrive/Documents/GitHub/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Amsterdam.csv'): WindowsPath('c:/Users/saiki/OneDrive/Documents/GitHub/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Amsterdam.csv'), WindowsPath('c:/Users/saiki/OneDrive/Documents/GitHub/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Atlanta.csv'): WindowsPath('c:/Users/saiki/OneDrive/Documents/GitHub/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Atlanta.csv'), WindowsPath('c:/Users/saiki/OneDrive/Documents/GitHub/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Boston.csv'): WindowsPath('c:/Users/saiki/OneDrive/Documents/GitHub/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Boston.csv'), WindowsPath('c:/Users/saiki/OneDrive/Documents/GitHub/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperatu

In [ ]:
df_max_all = {}
df_min_all = {}

for name in SET_USA:
    tmp_df = pd.read_csv(dct_name_file[name])
    tmp_df.index = pd.to_datetime(tmp_df["DATE"])
    df_max_all[name] = tmp_df["DAILY_MAX_TEMP"]
    df_min_all[name] = tmp_df["DAILY_MIN_TEMP"]
df_max_all = pd.DataFrame(df_max_all)
df_min_all = pd.DataFrame(df_min_all)
df_avg_all = (df_max_all + df_min_all) / 2
df_y = df_avg_all
df_max_all.head()

In [10]:
def plot_day_of_year(
    srs,
    is_day_of_year_frac=False,
    is_scatter=False,
    alpha=0.7,
    is_label=False,
    xlabel="Day of year",
    ylabel="Temperature",
    title="Temperature vs Day of year",
    axhline=[
        65,
    ],
    axvline_red=pd.DatetimeIndex(["2020-05-01", "2020-09-30"]).day_of_year.to_list(),
    axvline_blue=pd.DatetimeIndex(["2020-11-01", "2020-03-31"]).day_of_year.to_list(),
    ax=None,
):
    if is_day_of_year_frac:
        vec_t = (srs.index.day_of_year / (srs.index.is_leap_year + 365)).to_numpy()
    else:
        vec_t = srs.index.day_of_year.to_numpy()
    vec_y = srs.to_numpy()
    year_unique = srs.index.year.unique()
    cmap = plt.get_cmap("rainbow", len(year_unique))
    color_map = {year: cmap(i) for i, year in enumerate(year_unique)}
    if ax is None:
        fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 5))
    for year in year_unique:
        idx = srs.index.year == year
        if is_scatter:
            ax.scatter(
                vec_t[idx],
                vec_y[idx],
                s=1,
                color=color_map[year],
                alpha=alpha,
                label=year if is_label else None,
            )
        else:
            ax.plot(
                vec_t[idx],
                vec_y[idx],
                label=year,
                alpha=alpha,
                lw=1,
                c=color_map[year],
                linestyle="-.",
            )
    if axhline:
        for _ in axhline:
            ax.axhline(_, color="black", linestyle="--", lw=0.9)
    if axvline_red:
        for _ in axvline_red:
            ax.axvline(_, color="red", linestyle="--", lw=0.9)
    if axvline_blue:
        for _ in axvline_blue:
            ax.axvline(_, color="blue", linestyle="--", lw=0.9)
    ax.grid()
    if is_label:
        ax.legend()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return ax

In [11]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 10), sharex=True, sharey=True)
for station in SET_USA:
    plot_day_of_year(
        df_max_all[station],
        is_scatter=True,
        title="Daily max temp ~ day in year, obs from 13 stations in the US",
        ax=axes[0, 0],
    )
    plot_day_of_year(
        df_min_all[station],
        is_scatter=True,
        title="Daily min temp ~ day in year, obs from 13 stations in the US",
        ax=axes[0, 1],
    )
    plot_day_of_year(
        (df_max_all[station] + df_min_all[station]) / 2,
        is_scatter=True,
        title="Daily avg temp ~ day in year, obs from 13 stations in the US",
        ax=axes[1, 0],
    )
    plot_day_of_year(
        df_max_all[station] - df_min_all[station],
        is_scatter=True,
        title="Daily range temp ~ day in year, obs from 13 stations in the US",
        ax=axes[1, 1],
        axhline=None,
    )
for ax in axes.flatten():
    ax.set_xlim(0, 366)
    ax.set_ylim(-30, 120)
plt.tight_layout()

NameError: name 'df_max_all' is not defined

: 

In [ ]:
cmap = plt.get_cmap("rainbow", len(SET_USA))
color_map = {year: cmap(i) for i, year in enumerate(SET_USA)}
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 5))
for station in SET_USA:
    idx = df_max_all.index.month.isin(MONTH_CDD)
    ax.scatter(
        df_max_all[idx].index,
        df_max_all[idx][station],
        s=1,
        alpha=0.5,
        color=color_map[station],
        label=f"max, CDD months, {station}",
    )
    idx = df_min_all.index.month.isin(MONTH_HDD)
    ax.scatter(
        df_min_all[idx].index,
        df_min_all[idx][station],
        s=1,
        alpha=0.5,
        color=color_map[station],
        label=f"min, HDD months, {station}",
    )
plt.axhline(65, color="black", linestyle="--", lw=1)
plt.title("max in CDD months, min in HDD months")
# plt.legend()
plt.grid()
plt.tight_layout()